In [1]:
from IPython.display import display, HTML 
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

<font size="5" color="red">ch01_회귀모형 저장  </font>

- 데이터 소스 : 국토교통부 실거래가 공개시스템 https://rt.molit.go.kr/

In [4]:
import pandas as pd
import statsmodels.api as sm # 회귀모델
import joblib # pkl이나 joblib로 모델 저장, load

In [6]:
df = pd.read_csv('../data/trade_apt_api.csv', comment='#', encoding='cp949')
df.head(2)

,거래금액,건축년도,년,법정동,아파트,월,일,전용면적,지번,지역코드,층,해제사유발생일,해제여부
0,80000,2002,2021,신교동,신현(101동),8,16,84.82,6-13,11110,1,-,-
1,209000,2008,2021,사직동,광화문풍림스페이스본(106동),8,5,163.33,9-1,11110,13,-,-


In [7]:
df.sample()

,거래금액,건축년도,년,법정동,아파트,월,일,전용면적,지번,지역코드,층,해제사유발생일,해제여부
179,184000,2008,2021,사직동,광화문풍림스페이스본(101동~105동),3,20,159.01,9,11110,5,-,-


In [11]:
pd.options.mode.copy_on_write = True # 깊은 복사 실행 option
X = df[['건축년도', '전용면적', '층']]
X['const'] = 1
y = df['거래금액']
X.shape, y.shape

((318, 4), (318,))

In [12]:
model = sm.OLS(y, X).fit() # 회귀 모델
model.summary()
# R-squared (0.648) : X가 y를 설명해주는 정도(비율)
# Adj. R-squared : 수정된 r제곱 (설명도)
# Durbin-Watson : 자기 상관이 있는지 여부, 이상치는 2 이상
# coef : 추정 계수

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   거래금액   R-squared:                       0.648
Model:                            OLS   Adj. R-squared:                  0.644
Method:                 Least Squares   F-statistic:                     192.4
Date:                Wed, 09 Jul 2025   Prob (F-statistic):           8.54e-71
Time:                        10:12:31   Log-Likelihood:                -3777.5
No. Observations:                 318   AIC:                             7563.
Df Residuals:                     314   BIC:                             7578.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
건축년도        1925.6916    212.616      9.057      0.000    1507.360    2344.023
전용면적         962.1507     47.367     20.313      0.000     868.955    1055.347
층           2058.1524    417.716      4.927      0.000    1236.276    2880.028
const      -3.855e+06   4.25e+05     -9.069      0.000   -4.69e+06   -3.02e+06
==============================================================================
Omnibus:                       20.985   Durbin-Watson:                   1.352
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               42.734
Skew:                           0.345   Prob(JB):                     5.25e-10
Kurtosis:                       4.658   Cond. No.                     4.33e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.33e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [13]:
X.iloc[0], y[0]

(건축년도     2002.00
 전용면적       84.82
 층           1.00
 const       1.00
 Name: 0, dtype: float64,
 80000)

In [24]:
round(model.predict([2005, 103, 8, 1])[0]*10000, 1)

1214651869.0

In [16]:
format(121465, ',')

'121,465'

In [17]:
# 모델 저장
joblib.dump(model, '../model/ex1_apt_price_regression.joblib')

['../model/ex1_apt_price_regression.joblib']

In [25]:
def predict_apt_price(year, square, floor):
    loaded_model = joblib.load('../model/ex1_apt_price_regression.joblib')
    input_data = [[year, square, floor, 1]]
    result = round(loaded_model.predict(input_data)[0]*10000, 1)
    return format(result,',') + '원입니다'

In [26]:
year = int(input('건축 년도 ?'))
square = int(input('면적 (제곱미터) ?'))
floor = int(input('몇 층 ?'))
predict_apt_price(year, square, floor)

건축 년도 ?2003
면적 (제곱미터) ?192
몇 층 ?7


'2,011,870,666.1원입니다'